# Part I: UK Biobank research content, 2013–2025

This is the sole active content-analysis notebook. It combines the complete topic/category workflow with the independent panel-selection and assembly ledger in Part II. Exact pre-consolidation copies of both active source notebooks are preserved under `_archived/`; the earlier BERTopic and category-flow source notebooks remain archived there as well.

Part I prepares or reuses verified BERTopic assignments, builds shared Fields of Research (FOR), RCDC and topic aggregates, exports growth tables, and renders the main figure plus three supplementary figure families. Part II starts from a clean namespace, rebuilds the aggregates independently, records the publication-panel selection, redraws the selected figures, and exports the underlying panel tables and manifest.

The main figure places **A, FOR Level 4 composition** and **B, RCDC composition** side by side, with **C, thematic topic waves** below. Two supplements show annual category detail and classification coverage/breadth; a third shows topic-model robustness when its verified diagnostics are available. Figures use the shared project palette, Helvetica, bold left-aligned uppercase titles, and 500-DPI PNG plus PDF exports.

All analysis is restricted to **1 January 2013–31 December 2025 inclusive**. Topic modelling and figure generation are independent: completed topic assignments are checked before any modelling libraries are imported. Existing results are reused, while missing results trigger the preserved five-configuration, five-seed robustness workflow. Invalid or unverified results stop without a silent refit.

Growth tables rank FoRs, RCDC tags and cached topics by 2018–2025 compound annual growth in paper counts. They include recent growth and composition-share changes, with CSV, Excel and editable Word exports. A separate share-gain ranking retains emerging categories with small or zero baseline counts.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()

## 1. Reuse or compute topic results

Completed topic assignments are saved at **`output/bertopic/showcase_plus_id_topics.csv`**, relative to the project root: **23,441 publications, 67 topics, 2013–2025**. The matching `showcase_plus_id_topics.analysis_window.json` validates the saved results. The plots below reuse these assignments without fitting a model.

The same `output/bertopic/` directory contains `bertopic_document_topic_assignments.csv` (detailed assignments), `bertopic_topic_labels.csv` (labels), and the saved seed-robustness and cluster-persistence tables used by the diagnostic supplement.

Leave the defaults for a complete run. Set `RUN_TOPIC_MODELLING = False` to inspect the classification figures without fitting if results are absent. Set `FORCE_TOPIC_REFIT = True` only for an intentional refit.

The cache checks `output/bertopic/` and registered legacy CSV locations. Results must contain stable publication IDs, assignments, and a matching analysis-window provenance sidecar. Cached embeddings alone are not completed topic results. Embeddings and each successful parameter/seed fit are reused independently if a run is interrupted. The original SPECTER cache can be reused only after reproducing its ordered-corpus hash and matching the current documents; the clustering itself uses only the current analysis window.

In [ ]:
import json
import pandas as pd

# Completed assignments: output/bertopic/showcase_plus_id_topics.csv
TOPIC_RESULTS_DIR = P.OUTPUT / "bertopic"
RUN_TOPIC_MODELLING = os.environ.get("UKB_RUN_TOPIC_MODELLING", "1") != "0"
FORCE_TOPIC_REFIT = False

from utils import data_analysis_02_content_topics as T
TOPIC_RESULTS = T.ensure_topic_results(
    output_dir=TOPIC_RESULTS_DIR,
    train_if_missing=RUN_TOPIC_MODELLING,
    force=FORCE_TOPIC_REFIT,
)
if TOPIC_RESULTS is not None:
    topic_assignments = pd.read_csv(TOPIC_RESULTS)
    topic_column = "topics" if "topics" in topic_assignments else "topic"
    topic_labels = topic_assignments[topic_column].astype(str).str.strip()
    assigned = ~topic_labels.str.lower().isin(["outlier", "-1", "-1.0"])
    topic_provenance = json.loads(
        TOPIC_RESULTS.with_suffix(".analysis_window.json").read_text())
    print(f"Saved topic results: {len(topic_assignments):,} publications; "
          f"{topic_labels[assigned].nunique():,} topics.")
    print(f"Training years: {topic_provenance['training_min_year']}"
          f"–{topic_provenance['training_max_year']}.")
    print("Plotting data:", P.raw_path(TOPIC_RESULTS))

## 2. Prepare the shared figure data

FOR composition counts a publication once in each assigned Level 4 field, then divides by all paper–field assignments in that year. RCDC divides one unit per paper across its distinct tags. BERTopic supplies one topic per eligible publication after the existing outlier-reassignment procedure. Missing classifications remain outside composition denominators and are quantified in the coverage supplement.

The main figure uses leading categories selected by total in-window weight; the full distributions remain in the source tables. The topic-wave thickness is the annual share of **all non-outlier topic-assigned publications**, without renormalising the displayed topics to 100%. The vertical baseline is a display convention. Very small early cohorts are explicitly reported.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from utils.shared_style import PNG_DPI, load_style, savefig
from utils.shared_analysis_window import ANALYSIS_START_YEAR, ANALYSIS_END_YEAR
from utils import data_analysis_02_content_panels as C
from utils import data_analysis_02_content_exports as E

STYLE = load_style("02_content")
FLOW_MIN, FLOW_MAX = ANALYSIS_START_YEAR, ANALYSIS_END_YEAR
FLOW_YEARS = list(range(FLOW_MIN, FLOW_MAX + 1))
ARTIFACTS = P.ArtifactRegistry(P.TABLE_CONTENT)
D = C.build_panel_data(topic_results=TOPIC_RESULTS)
display(C.band_summary(D))

## 3. Fastest-growing fields, RCDC categories and topics

The default comparison is **2018–2025**, matching the main growth figure. Categories are ranked within each vocabulary by compound annual growth rate (CAGR) in **distinct publication counts**, requiring at least **10 papers in 2018** and positive growth. This avoids rankings dominated by changes such as 1→7 papers. At most 20 categories are shown per vocabulary; all categories and eligibility reasons remain in the complete CSV.

The tables also report 2024–2025 growth and changes in composition share. Counts are whole-paper counts for each category; a paper may carry multiple FoRs or RCDC tags. Composition shares retain the figure definitions: FoR paper–field assignments, fractionally weighted RCDC tags, and non-outlier topic assignments. **Percentage growth and percentage-point change in composition are different quantities.**

`content_largest_share_gains.csv` and the Excel share-gain sheets rank positive composition changes with at least 10 end-year papers, retaining categories with small or zero baseline counts. No growth ratio is assigned to a zero baseline. Missing classification years remain missing. Change the settings below to compare other years within 2013–2025.

Exports go to `output/tables/02_content/`: one ranked CSV and editable Word table per vocabulary, a combined Excel workbook, all-category statistics and methods. This cell uses `D` and never reruns BERTopic.


In [ ]:
from utils import data_analysis_02_content_growth as G

GROWTH_BASELINE_YEAR = 2018
GROWTH_END_YEAR = 2025
GROWTH_MIN_BASELINE_PAPERS = 10
GROWTH_TOP_N = 20

growth_tables = G.export_growth_tables(
    D, ARTIFACTS,
    baseline_year=GROWTH_BASELINE_YEAR,
    end_year=GROWTH_END_YEAR,
    min_baseline_papers=GROWTH_MIN_BASELINE_PAPERS,
    top_n=GROWTH_TOP_N,
)
for vocabulary, view in growth_tables["views"].items():
    print(f"{G.VOCABULARIES[vocabulary][0]}: {len(view)} fastest-growing categories")
    display(view)


## 4. Main figure: research composition and thematic waves

Panels are drawn directly from the shared aggregates. Rerunning this cell changes the figure without fitting a topic model. Full model labels, reviewed display descriptors and label sources are retained in `content_topic_labels.csv`.

In [ ]:
fig = C.figure_main(D, save=False)
main_stem = P.MAIN_FIGURE_STEMS[4]
if not D["topics"]["available"]:
    main_stem += "_incomplete"
    print("[WARN] Topic results are absent; this is an incomplete preview.")
ARTIFACTS.record_figures(savefig(fig, main_stem, style=STYLE))
display(fig)
plt.close(fig)

## 5. Supplementary figure: annual category detail

Annual shares of the twelve leading fields and RCDC tags, with percentage annotations and cell borders. The shared cool–cream–red colormap spans the full observed range within each panel. These are whole-vocabulary shares, so the twelve displayed rows need not sum to 100%.

In [ ]:
fig = C.figure_si_category_changes(D, save=False)
ARTIFACTS.record_figures(savefig(
    fig, "02_02_supplementary_figure_01_category_composition", style=STYLE))
display(fig)
plt.close(fig)

## 6. Supplementary figure: coverage and breadth

Coverage uses all papers in each year as denominator. Diversity is the exponential of Shannon entropy. Annual and cumulative category counts describe each vocabulary separately; different vocabularies have different granularity, and counts are not equivalent units. These descriptive measures also depend on corpus size.

In [ ]:
fig = C.figure_si_breadth_coverage(D, save=False)
ARTIFACTS.record_figures(savefig(
    fig, "02_03_supplementary_figure_02_coverage_and_breadth", style=STYLE))
display(fig)
plt.close(fig)

## 7. Supplementary figure: topic-model robustness

When the completed run supplies verified diagnostics, report variation across the five parameter configurations and five seeds, assignment stability, and final HDBSCAN cluster persistence. Diagnostic plots read saved tables and never fit a model. The representative seed is the assignment medoid under adjusted Rand agreement; the primary parameter score combines coherence, diversity, outlier rate, topic-count penalty and between-seed stability. Persistence is an additional diagnostic.

In [ ]:
diagnostics = C.load_topic_diagnostics(D)
if diagnostics is not None:
    fig = C.figure_si_topic_robustness(D, save=False)
    ARTIFACTS.record_figures(savefig(
        fig, "02_04_supplementary_figure_03_topic_robustness", style=STYLE))
    display(fig)
    plt.close(fig)
else:
    print("[SKIP] Verified topic-model robustness diagnostics are unavailable.")

## 8. Source tables, captions and manifest

Exports include full annual category distributions, denominators, category-share changes, displayed-band coverage, topic labels and analysis parameters. The caption and methods files document weighting, selection and the treatment of missing data. Figures are PNG/PDF; source tables are CSV, and growth rankings additionally have Excel and editable Word exports. JSON sidecars are used only for model-cache provenance.

The consolidated workflow removes the old hard-coded historical count comparisons and duplicate standalone figures. Earlier notebook implementations remain available in `_archived/` for reference.

In [ ]:
tables = E.export_content_tables(D, ARTIFACTS)
manifest, manifest_path = ARTIFACTS.save_manifest("02_content_manifest.csv")
print(f"Exported {len(ARTIFACTS.figure_paths)} figure files and "
      f"{len(ARTIFACTS.table_paths)} tables/captions; PNG resolution {PNG_DPI} DPI.")
print("Manifest:", P.raw_path(manifest_path))

---

Part II is executed independently from Part I so its aggregation and export checks cannot inherit notebook state.


In [ ]:
try:
    import matplotlib as _boundary_mpl
    import matplotlib.pyplot as _boundary_plt
    _boundary_plt.close("all")
    _boundary_mpl.rcdefaults()
except ImportError:
    pass
import warnings as _boundary_warnings
_boundary_warnings.resetwarnings()
from IPython import get_ipython as _boundary_get_ipython
_boundary_get_ipython().run_line_magic("reset", "-f")


# Part II: Publication-panel selection and assembly

This part preserves the former `02_content_99_all.ipynb` workflow. Its purpose is to make the publication selection auditable: it rebuilds all three content vocabularies from the canonical sources, records the banding rules and panel captions, redraws the selected figures, and writes the numerical tables behind them. The redraw is deliberate; using the same shared draw functions and export stems verifies that the assembled page is reproducible from a fresh namespace rather than pasted from Part I.

| vocabulary | analytical source | interpretation |
|---|---|---|
| BERTopic topics | verified per-publication assignments produced by Part I | themes learned from title and abstract text |
| FOR 2020 Level 4 | Showcase+ publisher-side classifications | disciplinary composition based on paper–field assignments |
| RCDC categories | Showcase+ NIH-derived tags | fractional condition/research-category composition |

The three vocabularies are intentionally shown together. A shift visible across them is evidence of a broad change in the literature; a shift visible in only one may reflect that vocabulary's construction. FOR papers can carry multiple fields, RCDC credit is divided over each paper's distinct tags, and BERTopic contributes one final non-outlier assignment per eligible publication.

The current band rules are data-driven and exported rather than embedded as unexplained literals. Each vocabulary has an explicit coverage target, minimum band share, maximum band count and remainder category. Because RCDC is much more granular than FOR, its grey remainder is expected to be larger; `panel_band_rules.csv` records the achieved coverage and number of pooled categories.

All panels are redrawn through `data_analysis_02_content_panels.py` under the shared `02_content` style. Legends use reviewed labels where available, the same light-grey remainder convention is applied across vocabularies, and the full distributions remain available in source tables even when only leading bands appear in a figure.


## 1. Setup

Part II uses the same shared `02_content` style as Part I. The namespace reset immediately before this section ensures that its successful redraw and table exports do not depend on objects left behind by Part I.


In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "utils").is_dir()
)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from utils import shared_paths as P
P.bootstrap()

import pandas as pd
from IPython.display import display

from utils.shared_style import load_style
from utils import data_analysis_02_content_panels as CP

STYLE = load_style("02_content")
FIG_DIR = Path(STYLE["savedir"])
if not FIG_DIR.is_absolute():
    FIG_DIR = P.ROOT / FIG_DIR
TABLE_DIR = P.TABLE_CONTENT
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("window  :", CP.FLOW_MIN, "-", CP.FLOW_MAX)
print("figures :", P.raw_path(FIG_DIR), "|", ", ".join(STYLE["formats"]),
      "at dpi", STYLE["dpi"])
print("tables  :", P.raw_path(TABLE_DIR))


## 2. Rebuild the aggregates from canonical sources

Everything drawn in this part comes from one freshly built dictionary, preventing rows of the same page from using different data vintages. FOR and RCDC are re-derived from the dated Showcase+ corpus. Topic assignments are read from the first complete, hash-bound result table under `output/bertopic/`; the detailed assignment table is authoritative when a compact convenience copy is stale.

This cell never silently refits BERTopic. Missing assignments produce an explicitly unavailable topic block, while malformed or unverified existing assignments raise before plotting.


In [ ]:
D = CP.build_panel_data()

### Banding thresholds, on the record

One row per vocabulary records the selection rule, number of displayed bands, achieved coverage and remainder size. The table is written to `output/tables/02_content/panel_band_rules.csv`, so the visual threshold is reproducible from data rather than justified after the fact.


In [ ]:
rules = CP.band_summary(D)
rules.to_csv(TABLE_DIR / "panel_band_rules.csv", index=False)
print(P.raw_path(TABLE_DIR / "panel_band_rules.csv"))
display(rules)

## 3. Panel selection, on the record

One row per panel records its figure, letter and caption. This keeps the manuscript's panel references aligned with the actual plotting functions and is exported beside the numerical source tables.


In [ ]:
rows = [{"figure": "main", "panel": letter, "caption": caption}
        for letter, caption in CP.MAIN_CAPTION.items()]
rows += [{"figure": f"si_{section}", "panel": letter, "caption": caption}
         for section, captions in CP.SI_CAPTIONS.items()
         for letter, caption in captions.items()]

selection = pd.DataFrame(rows)
selection.to_csv(TABLE_DIR / "panel_selection.csv", index=False)
print(f"{len(selection)} panels across {selection.figure.nunique()} figures -> "
      f"{P.raw_path(TABLE_DIR / 'panel_selection.csv')}\n")
display(selection)

## 4. Main-paper panel

The main page combines FOR Level 4 composition, fractionally weighted RCDC composition and centred thematic waves. Every year is normalised independently, so band width represents composition rather than publication volume. Each panel states or documents its own denominator, and the full category distributions are retained in CSV exports.

This redraw intentionally uses the same export stem as Part I. A difference would therefore expose non-deterministic aggregation or hidden notebook state rather than create a competing manuscript figure.


In [ ]:
fig_main = CP.figure_main(D)

## 5. Supplementary rank flow

The rank-flow supplement traces annual positions for the same leading FOR and RCDC categories used in the main figure. Rank and share answer different questions: the stacked bands show how much of the literature each category represents, while these trajectories show which categories crossed and when. Line width retains information about mean annual share.


In [ ]:
fig_si1 = CP.figure_si_rank_flow(D)

## 6. The tables behind the panels

The share matrices and the start-vs-end movement for each vocabulary. These are the
numbers a sentence in the paper should quote, rather than a value read off a band.

In [ ]:
written_tables = []
for key, stem in (("topics", "topics"), ("for", "for_l4"), ("rcdc", "rcdc")):
    block = D[key]
    if not block["available"]:
        print(f"{key}: source absent, no table written")
        continue
    for frame, name in (
        (block["share"].round(4).reset_index(), f"panel_{stem}_year_share.csv"),
        (block["band"].round(4).reset_index(), f"panel_{stem}_drawn_bands.csv"),
        (CP.start_end_table(block["share"], block["keep"]).reset_index(),
         f"panel_{stem}_start_vs_end.csv"),
    ):
        frame.to_csv(TABLE_DIR / name, index=False)
        written_tables.append(name)
    print(f"{key}: {block['n_bands']} bands, "
          f"{block['coverage_mean']:.1f}% mean cover -> 3 tables")

display(CP.start_end_table(D["for"]["share"], D["for"]["keep"]))

## 7. Output manifest

Every current `02_XX_*` figure file is listed with its size and modification time, making reruns auditable. Historical standalone `05a`–`07b` files are deliberately outside this manifest pattern and are not presented as outputs of the consolidated workflow.


In [ ]:
written = sorted(
    p for p in FIG_DIR.glob("02_[0-9][0-9]_*.*")
    if p.suffix.lower() in {".pdf", ".png", ".svg"}
)
manifest = pd.DataFrame([
    {"file": P.raw_path(p), "kb": p.stat().st_size // 1024,
     "modified": pd.Timestamp(p.stat().st_mtime, unit="s").round("s")}
    for p in written
])
manifest.to_csv(TABLE_DIR / "panel_manifest.csv", index=False)
print(f"{len(written)} figure files under {P.raw_path(FIG_DIR)} "
      f"({manifest.kb.sum() / 1024:.1f} MB) + {len(written_tables) + 3} tables\n")
display(manifest)